In [ ]:
# Custom modules
import util.review_util as review_util
import util.model_util as model_util

In [ ]:
text_reviews, star_reviews = review_util.get_reviews()
text_reviews = review_util.add_text_features(text_reviews)

In [ ]:
topic_model = model_util.get_bertopic()
review_topics, probs = topic_model.fit_transform(list(text_reviews.text))

# Add the predicted topics to your original dataframe
text_reviews['TopicID'] = review_topics

# topic_model.get_topic_info()

In [ ]:
topics_per_class = topic_model.topics_per_class(
    docs=list(text_reviews.text), classes=list(text_reviews.TopicID))

# review_util.view_review_topics(topics_per_class)

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(list(text_reviews.text))
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

tree = topic_model.get_topic_tree(hierarchical_topics)
# print(tree)

In [ ]:
# Force-assign the noise reviews to the closest topic
text_reviews_df = text_reviews.reset_index(drop=True)
new_topics = topic_model.reduce_outliers(list(text_reviews.text), review_topics, threshold=0.1)

# topic_model.merge_topics(docs=list(text_reviews.text),
#                          topics_to_merge=[[0,4], [2,3]])

# topic_model.get_topic_info()[['Topic', 'Count']]

In [ ]:
topic_model.update_topics(list(text_reviews.text), topics=new_topics,
                          ctfidf_model=model_util.get_ctfidf(), vectorizer_model=model_util.get_vectorizer())
# topic_model.get_topic_info()

In [ ]:
topics_per_class = topic_model.topics_per_class(docs=list(text_reviews.text),
                                                classes=list(text_reviews.TopicID))

# topic_model.visualize_topics_per_class(topics_per_class)

In [ ]:
# # Calculate mean stars for these new topics
topic_summary = text_reviews.groupby('TopicID')['stars'].mean()
# print(topic_summary)

In [ ]:
# topic_model.set_topic_labels(constants.topic_labels)
# topic_model.visualize_barchart(custom_labels=True)

In [ ]:
model_util.export_model(topic_model)

In [ ]:
# Visualization Idea(s)
# Are customers more likely to leave a text review when they are angry (1-star) or happy (5-star)?
# You can create a visualization comparing the Star Distribution of Text Reviews vs. Star Distribution of Non-Text Reviews. If 1-star ratings almost always include text, it proves that "Administrative Friction" (your Topic #2) is a high-engagement pain point.